In [ ]:
#CouseWork 3 Machine Leaning Group S
#A Machine Learning Model Using Breast Cancer dataset
#Group Details
#Kala Samuel              24/U/25045/PS        2400725045
#Njayaana Melissa Violet  24/U/1171            2400701171
#Kuboi Chebosis Lynn      24/U/04567/EVE       2400704567
#Mayanja Jessy Elijah     24/U/0680            2400700680
#Kuio David Mayang        24/U/21473/PS        2400721473

In [ ]:
#imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)

import shap
shap.initjs()
import lime
import lime.lime_tabular

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#loading data
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target   

print(f"Shape: {df.shape}")
df.head()

In [ ]:
#Data Exploration
df.info()
print()
print(f"Missing values in the dataset  : {df.isna().sum().sum()}")
df.describe()


In [ ]:
plt.figure(figsize=(20,18))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
# Remove multicollinearity 
corr_matrix = df.drop('target', axis=1).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

df = df.drop(columns=to_drop)

In [ ]:
# Distribution plot 
features = df.columns[:10]

df[features].hist(figsize=(12,8), bins= 20)
plt.suptitle("Feature Distribution")
plt.show()

In [ ]:
#Binary classification feauture
#since this is a Binary Classification problem
majority, minority = df["target"].value_counts(normalize=True)
print(f"Majority: {majority}")
print(f"Minotity: {minority}")

In [ ]:
#class  balance bar chart 
fig, ax = plt.subplots(figsize=(8, 6))
df["target"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Distribution of Classes")
ax.set_ylabel("Number of Samples")
ax.set_xlabel("Classes")

plt.show()

**Splitting Data**

In [ ]:
target = 'target'

X = df.drop(columns=target)    
y = df[target]      

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_test = pd.DataFrame(X_test, columns=X.columns)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")


**Training the Model** 
*Desicion Tree*

In [ ]:
model_dt = DecisionTreeClassifier(max_depth=6, random_state=42)

model_dt.fit(X_train, y_train)

**Evaluate Model Metrics**

In [ ]:
dt_pred = model_dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

**Train with Random Forest**

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)   

model_rf.fit(X_train, y_train)

**Evaluate Model Metrics**

In [ ]:
rf_pred = model_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

**TRAIN WITH DEEP LEARNING FEED-FORWARD NEURAL-NETWORK**

In [ ]:
model_nn =  MLPClassifier(
    hidden_layer_sizes=(30, 20),
    max_iter=1000,
    random_state=42
)

model_nn.fit(X_train, y_train)

**Evaluate the trained Model**

In [ ]:
nn_pred = model_nn.predict(X_test)

print("Neural Network Accuracy:", accuracy_score(y_test, nn_pred))

**Last Tasks**

❖Interpret the models’ decisions using SHAP.

❖Interpret the models’ decisions using LIME.

❖In your own words, compare the explanations of the three models for both
approaches.

In [ ]:
# Tree explainer and forest explainer

explainer_tree = shap.TreeExplainer(model_dt)
explainer_forest = shap.TreeExplainer(model_rf)  

shap_values_tree = explainer_tree.shap_values(X_test)
shap_values_tree=shap_values_tree[:,:,1]

shap_values_forest = explainer_forest.shap_values(X_test)
shap_values_forest=shap_values_forest[:,:,1]

# Summary plot for decision Tree

shap.summary_plot(shap_values_tree, X_test, show=False)
plt.title("SHAP Summary Plot for Decision Tree")
plt.tight_layout()
plt.show()

# Summary plot for Random Forest
shap.summary_plot(shap_values_forest, X_test, show =False)
plt.title("SHAP Summary Plot for Random Forest")
plt.tight_layout()
plt.show()

## SHAP for Neural Networks

Since neural networks are non-linear and do not have built-in tree structures, SHAP values are computed using KernelExplainer, which approximates feature contributions by sampling predictions. This makes it computationally expensive but model-agnostic.

Because the model was trained on scaled data, SHAP analysis is performed on a small sample of scaled test data for efficiency.

In [ ]:
# SHAP FOR NEURAL NETWORK

# Small background sample
background = X_train[:100]

explainer_nn = shap.KernelExplainer(model_nn.predict_proba, background)

# Explain a small subset (VERY important for speed)
X_sample = X_test[:20]
X_sample_df = pd.DataFrame(X_sample, columns=X.columns)

shap_values_nn = explainer_nn.shap_values(X_sample)

if isinstance(shap_values_nn, list):
    shap_nn = shap_values_nn[1]
elif len(np.array(shap_values_nn).shape) == 3:
    shap_nn = shap_values_nn[:, :, 1]
else:
    shap_nn = shap_values_nn

shap.summary_plot(shap_nn, X_sample_df, show=False)
plt.title("SHAP Summary Plot for Neural Network")
plt.tight_layout()
plt.show()

The neural network shows a more distributed pattern of feature importance compared to tree-based models. This reflects its ability to capture complex non-linear interactions across many variables rather than relying on a few dominant features.

### LIME FOR ALL MODEL

#### LIME Interpretation

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by approximating the model locally using a simpler interpretable model. It highlights which features contributed positively or negatively to a specific prediction.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
from IPython.display import display, HTML

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=X.columns,
    class_names=['malignant', 'benign'],
    mode='classification'
)

In [ ]:
exp_rf = lime_explainer.explain_instance(
    X_test.iloc[0].values,
    model_rf.predict_proba
)

display(HTML(exp_rf.as_html()))

The LIME plot shows how features like mean, radius, mean,texture, radius error contribute positively towrds this specific instance being preducted as malignant.

In [ ]:
exp_dt = lime_explainer.explain_instance(
    X_test.iloc[0].values,
    model_dt.predict_proba
)


display(HTML(exp_dt.as_html()))

In [ ]:
exp_nn = lime_explainer.explain_instance(
    X_test.iloc[0].values,
    lambda x: model_nn.predict_proba(scaler.transform(x))
)


display(HTML(exp_nn.as_html()))

LIME inteprets individual prediction of teh model. For the selected instance, the explanation shows the contribution of each feature toward the predicted class. The length and color intensity of the bars represent the strength and direction of each feature’s influence.

## SHAP vs LIME vs Models

SHAP provides a globally consistent and theoretically grounded explanation of feature contributions based on game theory, while LIME provides local explanations by approximating model behavior around a single instance.

Tree-based models (Decision Tree and Random Forest) show concentrated feature importance, meaning a small number of features dominate predictions. In contrast, the Neural Network distributes importance across many features, reflecting its ability to learn complex non-linear relationships.

Overall, SHAP provides more stable global interpretability, while LIME offers intuitive local explanations. Together, they provide complementary insights into model behavior.